In [3]:
### Simple Retrieval-Augmented Generation
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.document_loaders import TextLoader

RuntimeError: Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
module 'tensorflow._api.v2.compat.v2.__internal__' has no attribute 'register_load_context_function'

In [ ]:
## KB Creation
loader = TextLoader("./TestDocument.txt")
doc = loader.load()

In [ ]:
# Splitting document into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(doc)
print(f"Generated {len(chunks)} chunks from your document(s).")

In [ ]:
## Embedding/Indexing
model = SentenceTransformer("all-MiniLM-L6-v2")

og_embeddings = model.encode(chunks)
vector_db = dict(enumerate(og_embeddings.flatten()))
print(og_embeddings)
print(vector_db)

RuntimeError: Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [ ]:
## Retrieval

# Query for doc
def query_text(indices: list) -> list:
    output = [vector_db[index] for index in indices]
    return output

indices = [4,10]
print(query_text(indices))

In [ ]:
def retrieve(query, top_k):
    embeddings = model.encode(query)
    similarities = cosine_similarity(og_embeddings, embeddings)
    print(similarities)
    sim_inds = np.argsort(-similarities)
    top_k_inds = sim_inds[:top_k]
    retrieval = query_text(top_k_inds)
    return retrieval

In [ ]:
query = "Who is Dave Arneson?"
retrieval = retrieve(query, 5)

In [ ]:
## Generation
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

set_seed(125)

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2", pad_token_id=tokenizer.eos_token_id).to(device)

In [ ]:
prompt = f"Here is data about a subject: {retrieval}. Please answer this question using that data: {query}"
model_inputs = tokenizer(prompt, return_tensors='pt').to(device)
outputs = model.generate(
    **model_inputs, 
    max_new_tokens=150,
    do_sample=True,
    top_p=0.9,
    top_k=50,
    temperature=0.6
    )
tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [ ]:
## Testing
